In [1]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-4B"
LORA_PATH = "/home/avinash/Projects/Custom-LN-Translator-Training/Model-Checkpoints/checkpoint-400"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(
    base_model,
    LORA_PATH,
)


/home/avinash/.pyenv/versions/llm-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]/home/avinash/.pyenv/versions/llm-env/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 3/3 [01:16<00:00, 25.58s/it]


In [5]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # needed for packing

In [6]:
jp_text =  """
「高揚してるのか、俺にもよく分からん」
"""
    
messages = [
    {
        "role": "system",
        "content": "You are a careful literary translation model. Translate faithfully, preserve tone, punctuation, names, and paragraph structure."
    },
    {
        "role": "user",
        "content": jp_text
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
    **inputs,
    do_sample=False,
    temperature=None,
    max_new_tokens=300,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)


print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/home/avinash/.pyenv/versions/llm-env/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


system
You are a careful literary translation model. Translate faithfully, preserve tone, punctuation, names, and paragraph structure.
user

「高揚してるのか、俺にもよく分からん」

assistant
<think>

</think>

"I don't know if you're just excited, but I don't get it."

I don't know if you're just excited, but I don't get it."

I don't know if you're just excited, but I don't get it."


In [20]:
import gc
import torch


# 2. Invoke Python's garbage collector to reclaim unreferenced memory
gc.collect()

# 3. Clear the PyTorch CUDA memory cache
torch.cuda.empty_cache()   

In [7]:
import random
from pathlib import Path
import torch

ROOT = Path("/home/avinash/Projects/Custom-LN-Translator-Training/Assets/Novels")
OUTPUT_FILE = Path("novel_probe_report.txt")

CHAPTERS_PER_NOVEL = 2
MAX_PARAGRAPHS = 1
MAX_NEW_TOKENS = 128

# Tune this for your VRAM
BATCH_SIZE = 2

SYSTEM_PROMPT = (
    "You are a careful literary translation model. "
    "Translate faithfully, preserve tone, punctuation, "
    "names, and paragraph structure."
)

samples = []

print("Collecting samples...")

for novel_dir in sorted(ROOT.iterdir()):

    if not novel_dir.is_dir():
        continue

    jp_output_dir = novel_dir / "JP-Output"

    if not jp_output_dir.exists():
        continue

    chapter_files = sorted(jp_output_dir.glob("*.txt"))

    if not chapter_files:
        continue

    selected = random.sample(
        chapter_files,
        min(CHAPTERS_PER_NOVEL, len(chapter_files))
    )

    for chapter_file in selected:

        try:
            text = chapter_file.read_text(
                encoding="utf-8",
                errors="ignore"
            ).strip()

            paragraphs = [
                p.strip()
                for p in text.split("\n\n")
                if p.strip()
            ]

            jp_sample = "\n\n".join(
                paragraphs[:MAX_PARAGRAPHS]
            )

            messages = [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content": jp_sample,
                }
            ]

            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            samples.append({
                "novel": novel_dir.name,
                "chapter": chapter_file.name,
                "jp": jp_sample,
                "prompt": prompt,
            })

        except Exception as e:
            print(f"Failed collecting {chapter_file}: {e}")

print(f"Collected {len(samples)} samples")

# --------------------------------------------------
# Batched inference
# --------------------------------------------------

translations = []

for start in range(0, len(samples), BATCH_SIZE):

    batch = samples[start:start + BATCH_SIZE]

    prompts = [x["prompt"] for x in batch]

    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        # 3. Clear the PyTorch CUDA memory cache
        torch.cuda.empty_cache()  

        outputs = model.generate(
            **inputs,
            do_sample=False,
            repetition_penalty=1.1,
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_lengths = inputs["attention_mask"].sum(dim=1)

    for i, sample in enumerate(batch):

        generated = outputs[i][input_lengths[i]:]

        translation = tokenizer.decode(
            generated,
            skip_special_tokens=True,
        )

        translations.append({
            **sample,
            "translation": translation,
        })

    print(
        f"Processed "
        f"{min(start+BATCH_SIZE, len(samples))}"
        f"/{len(samples)}"
    )

# --------------------------------------------------
# Report writing
# --------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as report:

    current_novel = None

    for item in translations:

        if item["novel"] != current_novel:

            current_novel = item["novel"]

            report.write("\n")
            report.write("#" * 120 + "\n")
            report.write(f"NOVEL: {current_novel}\n")
            report.write("#" * 120 + "\n\n")

        report.write("-" * 120 + "\n")
        report.write(f"CHAPTER: {item['chapter']}\n")
        report.write("-" * 120 + "\n\n")

        report.write("[JP INPUT]\n\n")
        report.write(item["jp"])
        report.write("\n\n")

        report.write("[MODEL OUTPUT]\n\n")
        report.write(item["translation"])
        report.write("\n\n\n")

print(f"Saved report -> {OUTPUT_FILE}")

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Collected 22 samples


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Processed 2/22


KeyboardInterrupt: 